In [ ]:
# =========================================================
# Notebook 05 — Feature / Artefact Diagnostics
# =========================================================
#
# Purpose:
# Perform a lightweight diagnostic check to compare the
# feature expectations of a trained XGBoost model against the
# columns available in the final engineered panel.
#
# Inputs:
# - final_xgb_model.joblib
# - school_panel_final.parquet
#
# Stored in:
# - data/processed/
#
# Outputs:
# - Console diagnostic output only
#
# Role in the pipeline:
# This notebook is a quality-assurance / diagnostic step. It
# is not required for the core modelling pipeline to run, but
# it helps verify that the trained model’s expected feature
# space aligns with the final engineered dataset.
#
# Reproducibility note:
# The file paths in this notebook are currently configured for
# the author’s local machine. These should be updated if the
# project is run in a different environment.
#
# Important:
# This notebook was developed on a local Windows environment.
# Users reproducing the pipeline should replace absolute paths
# with environment-specific paths or relative project paths.
#
# Diagnostic note:
# This notebook is designed as a manual inspection aid rather
# than a formal evaluation stage. It checks whether broad
# feature families (e.g. attendance, workforce, disadvantage)
# appear both in the model’s expected feature list and in the
# engineered panel.
# =========================================================

import joblib
import pandas as pd
import os
import xgboost as xgb

# =========================================================
# Configuration
# =========================================================
#
# This section defines the paths used to load:
# - the trained XGBoost model
# - the final engineered panel
#
# If reproducing this notebook on another machine, update
# these paths before running the notebook.
# =========================================================

# --- CONFIG ---
base_path = r'C:\Users\kiero\Documents\msc-dissertation\data\processed'
model_path = os.path.join(base_path, 'delta_xgb_model.joblib')
data_path = os.path.join(base_path, 'school_panel_final.parquet')

print("🔍 STARTING DIAGNOSTICS...\n")

# =========================================================
# Step 1 — Load Model Feature Expectations
# =========================================================
#
# This step loads the trained XGBoost model and extracts the
# feature names it expects at inference time.
#
# If `feature_names_in_` is unavailable, the code falls back
# to booster-level feature names.
# =========================================================

# 1. LOAD MODEL FEATURES
try:
    model = joblib.load(model_path)
    if hasattr(model, "feature_names_in_"):
        model_features = model.feature_names_in_.tolist()
    else:
        model_features = model.get_booster().feature_names
    print(f" Model Loaded. It expects {len(model_features)} features.")
except Exception as e:
    print(f"❌ Model Load Failed: {e}")
    model_features = []

# =========================================================
# Step 2 — Load Final Panel Columns
# =========================================================
#
# This step loads the engineered panel and extracts the full
# list of available columns for comparison.
# =========================================================

# 2. LOAD DATA COLUMNS
try:
    df = pd.read_parquet(data_path)
    data_columns = df.columns.tolist()
    print(f" Data Loaded. It has {len(data_columns)} columns.")
except Exception as e:
    print(f"❌ Data Load Failed: {e}")
    data_columns = []

print("\n" + "="*40)
print(" CRITICAL MATCH CHECK")
print("="*40)

# =========================================================
# Step 3 — Search for Key Feature Families
# =========================================================
#
# This step performs a broad concept-level comparison between:
# - model feature names
# - panel column names
#
# Categories checked:
# - absence / attendance
# - teachers / workforce
# - disadvantage
#
# This is not a strict schema validation routine; it is a
# quick manual diagnostic to confirm that expected feature
# families are present.
# =========================================================


# 3. SEARCH FOR KEY CONCEPTS
keywords = {
    "ABSENCE": ['ABS', 'PERS', 'ATTEND', 'UNAUTH', 'SESS'],
    "TEACHERS": ['TEACH', 'PTR', 'WORK', 'PERC', 'HEADCOUNT'],
    "DISADVANTAGE": ['FSM', 'DISAD', 'POV', 'NUM']
}

for category, keys in keywords.items():
    print(f"\n--- {category} ---")
    
    # Check Model
    model_matches = [f for f in model_features if any(k in f.upper() for k in keys)]
    print(f"   Model expects: {model_matches[:5]} ... (Total: {len(model_matches)})")
    
    # Check Data
    data_matches = [c for c in data_columns if any(k in c.upper() for k in keys)]
    print(f"   Data contains: {data_matches[:5]} ... (Total: {len(data_matches)})")

print("\n" + "="*40)
print("\nDiagnostic check complete.")


# =========================================================
# Outputs Produced
# =========================================================
#
# This notebook does not create saved artefacts. Its output is
# a printed diagnostic summary intended for manual inspection.
#
# Use case:
# - confirm broad alignment between model feature families and
#   engineered dataset columns
# - investigate potential feature mismatch issues before model
#   deployment or further evaluation
# =========================================================

🔍 STARTING DIAGNOSTICS...

 Model Loaded. It expects 232 features.
 Data Loaded. It has 262 columns.

🔎 CRITICAL MATCH CHECK

--- ABSENCE ---
   Model expects: ['TEACHERS WITH AT LEAST ONE PERIOD OF SICKNESS ABSENCE (%)', 'TOTAL NUMBER OF DAYS LOST TO SICKNESS ABSENCE', 'AVERAGE (MEAN) NUMBER OF DAYS LOST TO TEACHER SICKNESS ABSENCE (ONLY TEACHERS IN SCHOOL TAKING SICKNESS ABSENCE)', 'AVERAGE NUMBER OF DAYS LOST TO TEACHER SICKNESS ABSENCE (ALL TEACHERS IN SCHOOL)', 'ABSENCE_RATE'] ... (Total: 15)
   Data contains: ['TEACHERS WITH AT LEAST ONE PERIOD OF SICKNESS ABSENCE (%)', 'TOTAL NUMBER OF DAYS LOST TO SICKNESS ABSENCE', 'AVERAGE (MEAN) NUMBER OF DAYS LOST TO TEACHER SICKNESS ABSENCE (ONLY TEACHERS IN SCHOOL TAKING SICKNESS ABSENCE)', 'AVERAGE NUMBER OF DAYS LOST TO TEACHER SICKNESS ABSENCE (ALL TEACHERS IN SCHOOL)', 'ABSENCE_RATE'] ... (Total: 15)

--- TEACHERS ---
   Model expects: ['TOTAL SCHOOL WORKFORCE (HEADCOUNT)', 'TOTAL NUMBER OF CLASSROOM TEACHERS (HEADCOUNT)', 'TOTAL NUMB